# Étape 5b - Agrégations : Silver -> Gold

Documentation de chaque table : `docs/data-dictionary/gold.md`.

Calcule des indicateurs qui répondent aux questions métier du Jour 1
(`UC01_questions_decisionnelles.md`), avec les formules du dictionnaire de KPI
(`UC01_dictionnaire_KPI.md`) :

| KPI | Formule |
|---|---|
| KPI-01 Chiffre d'affaires | somme de `montant_net_paye` (`ss_net_paid` : remises et coupons déduits, hors taxes) |
| KPI-02 Transactions | nombre de tickets distincts |
| KPI-03 Panier moyen | CA ÷ transactions |
| KPI-04 Quantités vendues | somme de `quantite` |
| KPI-05 Taux de remise | somme des remises ÷ somme des prix catalogue × 100 |
| KPI-06 Évolution du CA | (CA N − CA N-1) ÷ CA N-1 × 100 |
| Marge | somme de `marge` (`ss_net_profit`), taux = marge ÷ CA × 100 |

- Entrée : `data/silver/sales_enriched/`.
- Sortie : `data/gold/<indicateur>/` en Parquet.
- Montants en `decimal`, arrondis à 2 décimales.

> Les ventes viennent d'un **échantillon de 10 % des tickets** : les montants sont
> environ 10 fois plus petits que la réalité, mais les parts, taux, paniers moyens
> et évolutions restent représentatifs.

**Organisation du notebook**

| Partie | Rôle |
|---|---|
| 0. Paramètres | Chemins et session Spark lus dans `config/parametres.json` ; journal ; **lecture** de `sales_enriched` |
| `indicateurs`, `ratios`, `ecrire_gold` | **Fonctions** : mesures communes, ratios calculés sur les sommes, **écriture** avec provenance et export du schéma |
| 1 à 4 | **Transformations** : une agrégation par question métier |
| 5 | **Contrôles** (totaux Gold = Silver) et **résultats** : `outputs/bilan_gold.json` |

## 0. Paramètres, chemins, session Spark et journal

Tous les paramètres et chemins viennent de `config/parametres.json` : aucun chemin n'est écrit en dur dans ce notebook.
Le journal d'exécution est ajouté dans `outputs/logs/<notebook>.log`.

In [1]:
import json
import os
from datetime import datetime

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

# ---------- Parametres et chemins (config/parametres.json) ----------
RACINE = os.getcwd()
while not os.path.exists(os.path.join(RACINE, "config", "parametres.json")):
    if os.path.dirname(RACINE) == RACINE:
        raise RuntimeError("config/parametres.json introuvable : lancer le notebook depuis le projet")
    RACINE = os.path.dirname(RACINE)
with open(os.path.join(RACINE, "config", "parametres.json")) as f:
    PARAMS = json.load(f)


def chemin(cle):
    """Chemin absolu d'une entree de PARAMS["chemins"] (relative a la racine, ~ accepte)."""
    return os.path.normpath(os.path.join(RACINE, os.path.expanduser(PARAMS["chemins"][cle])))


LOCAL = "file://"  # sans ce prefixe, Spark irait chercher les fichiers dans HDFS (fs.defaultFS)
SOURCE = chemin("source_tpcxbb")
SILVER = LOCAL + chemin("silver")
GOLD = LOCAL + chemin("gold")
OUTPUTS = chemin("outputs")
DOSSIER_SCHEMAS = chemin("schemas")
# store_sales : echantillon ou fichier complet selon le parametre "echantillon.actif"
FICHIER_VENTES = (chemin("echantillon_ventes") if PARAMS["echantillon"]["actif"]
                  else os.path.join(SOURCE, "store_sales", "store_sales.dat"))
os.makedirs(chemin("logs"), exist_ok=True)

# ---------- Journal d'execution ----------
FICHIER_JOURNAL = os.path.join(chemin("logs"), "agregations_gold.log")


def journal(*message):
    ligne = datetime.now().strftime("%Y-%m-%d %H:%M:%S") + " | agregations_gold | " + " ".join(str(m) for m in message)
    print(ligne)
    with open(FICHIER_JOURNAL, "a") as f:
        f.write(ligne + "\n")


# ---------- Session Spark (parametres "spark") ----------
constructeur = SparkSession.builder.appName("TP3-Agregations-Gold").master(PARAMS["spark"]["master"])
for cle, valeur in PARAMS["spark"]["config"].items():
    constructeur = constructeur.config(cle, valeur)
spark = constructeur.getOrCreate()
spark.sparkContext.setLogLevel(PARAMS["spark"]["niveau_journal"])

journal("debut", "| echantillon" if PARAMS["echantillon"]["actif"] else "| fichier complet",
        "| Spark", spark.version, PARAMS["spark"]["master"])

ventes = spark.read.parquet(SILVER + "/sales_enriched").cache()
print("Lignes de vente :", ventes.count())

resultats = {}


def indicateurs():
    """Mesures communes a toutes les agregations."""
    return [
        F.round(F.sum("montant_net_paye"), 2).alias("chiffre_affaires"),
        F.countDistinct("ticket_id").alias("nb_transactions"),
        F.sum("quantite").alias("quantites_vendues"),
        F.round(F.sum("marge"), 2).alias("marge"),
        F.round(F.sum("montant_remise"), 2).alias("remises"),
        F.round(F.sum("montant_catalogue"), 2).alias("montant_catalogue"),
        F.count("*").alias("nb_lignes"),
    ]


def ratios(df):
    """Ratios calcules a partir des sommes (et non en moyennant des taux ligne a ligne)."""
    return (df
        .withColumn("panier_moyen", F.round(F.col("chiffre_affaires") / F.col("nb_transactions"), 2))
        .withColumn("taux_marge_pct", F.round(F.col("marge") / F.col("chiffre_affaires") * 100, 2))
        .withColumn("taux_remise_pct", F.round(F.col("remises") / F.col("montant_catalogue") * 100, 2)))


def ecrire_gold(df, nom, granularite):
    """Ecrit une table Gold avec sa provenance, puis exporte son schema de reference."""
    df = (df
        .withColumn("_sources_silver", F.lit("silver/sales_enriched"))
        .withColumn("_traitement", F.lit("notebooks/transformations/agregations_gold.ipynb"))
        .withColumn("_date_traitement", F.current_timestamp()))
    df.write.mode("overwrite").parquet(GOLD + "/" + nom)
    relu = spark.read.parquet(GOLD + "/" + nom)
    with open(os.path.join(DOSSIER_SCHEMAS, f"gold_{nom}.json"), "w") as f:
        json.dump(json.loads(relu.schema.json()), f, indent=1)
    resultats[nom] = {"granularite": granularite, "lignes": relu.count()}
    print(f"Gold {nom} : {relu.count()} ligne(s) - granularite : {granularite}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 17:30:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-11-18 17:30:24 | agregations_gold | debut | echantillon | Spark 4.3.0 local[2]


Lignes de vente : 66593


## 1. CA mensuel et évolution par rapport à l'année précédente

- Question P1 : « Comment évolue le chiffre d'affaires chaque mois et par rapport à l'année d'avant ? »
- **Granularité** : une ligne = un mois (année, mois).
- **Règle** : l'évolution compare au même mois de l'année N-1 (KPI-06). Elle est vide pour 2001, qui n'a pas d'année précédente.
- **Limite** : décembre 2005 est incomplet (dernière vente le 13/12/2005).

In [2]:
ca_mensuel = ratios(ventes.groupBy("annee", "mois").agg(*indicateurs()))

meme_mois = Window.partitionBy("mois").orderBy("annee")
ca_mensuel = (ca_mensuel
    .withColumn("ca_annee_precedente", F.lag("chiffre_affaires").over(meme_mois))
    .withColumn("evolution_ca_pct",
                F.round((F.col("chiffre_affaires") - F.col("ca_annee_precedente"))
                        / F.col("ca_annee_precedente") * 100, 2))
    .withColumn("mois_incomplet", (F.col("annee") == 2005) & (F.col("mois") == 12))
    .orderBy("annee", "mois"))

ca_mensuel.select("annee", "mois", "chiffre_affaires", "nb_transactions", "panier_moyen",
                  "ca_annee_precedente", "evolution_ca_pct", "mois_incomplet").show(60)
ecrire_gold(ca_mensuel, "ca_mensuel", "un mois (annee, mois)")

+-----+----+----------------+---------------+------------+-------------------+----------------+--------------+
|annee|mois|chiffre_affaires|nb_transactions|panier_moyen|ca_annee_precedente|evolution_ca_pct|mois_incomplet|
+-----+----+----------------+---------------+------------+-------------------+----------------+--------------+
| 2001|   1|      1933073.91|            153|    12634.47|               NULL|            NULL|         false|
| 2001|   2|      1772121.45|            138|    12841.46|               NULL|            NULL|         false|
| 2001|   3|      2188732.07|            153|    14305.44|               NULL|            NULL|         false|
| 2001|   4|      1895642.79|            148|    12808.40|               NULL|            NULL|         false|
| 2001|   5|      1879177.28|            153|    12282.20|               NULL|            NULL|         false|
| 2001|   6|      1871513.71|            148|    12645.36|               NULL|            NULL|         false|
|

Gold ca_mensuel : 60 ligne(s) - granularite : un mois (annee, mois)


In [3]:
# Vue annuelle, plus lisible pour la direction
ca_annuel = (ratios(ventes.groupBy("annee").agg(*indicateurs()))
    .withColumn("evolution_ca_pct",
                F.round((F.col("chiffre_affaires") - F.lag("chiffre_affaires").over(Window.orderBy("annee")))
                        / F.lag("chiffre_affaires").over(Window.orderBy("annee")) * 100, 2))
    .orderBy("annee"))
ca_annuel.select("annee", "chiffre_affaires", "evolution_ca_pct", "nb_transactions", "panier_moyen",
                 "marge", "taux_marge_pct", "taux_remise_pct").show()
ecrire_gold(ca_annuel, "ca_annuel", "une annee")

+-----+----------------+----------------+---------------+------------+------------+--------------+---------------+
|annee|chiffre_affaires|evolution_ca_pct|nb_transactions|panier_moyen|       marge|taux_marge_pct|taux_remise_pct|
+-----+----------------+----------------+---------------+------------+------------+--------------+---------------+
| 2001|     23404661.59|            NULL|           1800|    13002.59|-11153690.51|        -47.66|          50.05|
| 2002|     23037968.91|           -1.57|           1799|    12805.99|-11109564.43|        -48.22|          50.26|
| 2003|     23591448.60|            2.40|           1799|    13113.65|-11101186.91|        -47.06|          49.67|
| 2004|     23713771.57|            0.52|           1803|    13152.40|-11340198.12|        -47.82|          50.05|
| 2005|     22004326.01|           -7.21|           1709|    12875.56|-10536612.13|        -47.88|          49.85|
+-----+----------------+----------------+---------------+------------+----------

Gold ca_annuel : 5 ligne(s) - granularite : une annee


## 2. CA et marge par catégorie de produits

- Questions P4 : « Quels produits et quelles catégories génèrent le plus de chiffre d'affaires ? » ; P2 : « Quelle marge fait chaque catégorie ? »
- **Granularité** : une ligne = une catégorie.
- **Règle** : `part_ca_pct` = CA de la catégorie ÷ CA total × 100.

In [4]:
ca_categorie = (ratios(ventes.groupBy("categorie_id", "categorie").agg(
        *indicateurs(), F.countDistinct("produit_sk").alias("nb_produits_vendus")))
    .withColumn("part_ca_pct", F.round(F.col("chiffre_affaires")
                                       / F.sum("chiffre_affaires").over(Window.partitionBy()) * 100, 2))
    .orderBy(F.desc("chiffre_affaires")))

ca_categorie.select("categorie", "chiffre_affaires", "part_ca_pct", "marge", "taux_marge_pct",
                    "taux_remise_pct", "quantites_vendues", "nb_produits_vendus").show(truncate=False)
ecrire_gold(ca_categorie, "ca_marge_par_categorie", "une categorie de produits")

+------------------------+----------------+-----------+------------+--------------+---------------+-----------------+------------------+
|categorie               |chiffre_affaires|part_ca_pct|marge       |taux_marge_pct|taux_remise_pct|quantites_vendues|nb_produits_vendus|
+------------------------+----------------+-----------+------------+--------------+---------------+-----------------+------------------+
|Music                   |52109109.95     |45.02      |-25089209.40|-48.15        |49.99          |1530364          |7164              |
|Books                   |30720062.81     |26.54      |-14752808.59|-48.02        |50.04          |898730           |4688              |
|Movies & TV             |7342177.59      |6.34       |-3580572.60 |-48.77        |50.35          |217332           |1121              |
|Clothing & Accessories  |5768943.01      |4.98       |-2565727.59 |-44.47        |49.27          |165362           |528               |
|Electronics             |5682478.05     

Gold ca_marge_par_categorie : 9 ligne(s) - granularite : une categorie de produits


In [5]:
# Top 10 des produits par CA (question P4, niveau produit)
top_produits = (ventes.groupBy("produit_sk", "produit_id", "nom_produit", "categorie").agg(*indicateurs())
    .orderBy(F.desc("chiffre_affaires")).limit(10))
top_produits.select("nom_produit", "categorie", "chiffre_affaires", "quantites_vendues", "marge").show(truncate=40)
ecrire_gold(top_produits, "top10_produits", "un produit (10 premiers par CA)")

+----------------------------------------+----------------------+----------------+-----------------+----------+
|                             nom_produit|             categorie|chiffre_affaires|quantites_vendues|     marge|
+----------------------------------------+----------------------+----------------+-----------------+----------+
|   vwPR6l3UPyCIv5bljZbIG9k9YHLajogyqKbd6|        Home & Kitchen|      1616303.86|            46265|-747702.21|
|                         6VxbaI6thqY49zO|                 Music|      1469375.01|            42959|-746277.56|
|                                    Z2kc|           Electronics|      1404863.20|            40822|-671217.02|
|bwTLbeGP8aT6bW7YjgxmDJuCUTlwG1Fz78Vew...|                 Music|      1289141.51|            38970|-658302.66|
|                             bKIpo9aB7pb|                 Books|       402054.46|             9632|-103591.12|
|           ruqCzl5gIq5XGykXpKxbGQ9No16VY|           Electronics|       374321.55|            10069|-151

Gold top10_produits : 10 ligne(s) - granularite : un produit (10 premiers par CA)


## 3. CA et marge par magasin

- Question P2 : « Quelle marge fait chaque magasin ? »
- **Granularité** : une ligne = un magasin.

In [6]:
ca_magasin = (ratios(ventes.groupBy("magasin_sk", "magasin_id", "nom_magasin", "ville", "etat")
                          .agg(*indicateurs()))
    .withColumn("part_ca_pct", F.round(F.col("chiffre_affaires")
                                       / F.sum("chiffre_affaires").over(Window.partitionBy()) * 100, 2))
    .orderBy(F.desc("chiffre_affaires")))

ca_magasin.select("nom_magasin", "ville", "etat", "chiffre_affaires", "part_ca_pct", "nb_transactions",
                  "panier_moyen", "marge", "taux_marge_pct").show(truncate=False)
ecrire_gold(ca_magasin, "ca_marge_par_magasin", "un magasin")

+------------+------------+----+----------------+-----------+---------------+------------+-----------+--------------+
|nom_magasin |ville       |etat|chiffre_affaires|part_ca_pct|nb_transactions|panier_moyen|marge      |taux_marge_pct|
+------------+------------+----+----------------+-----------+---------------+------------+-----------+--------------+
|Sherwood    |Mount Vernon|CA  |10140403.47     |8.76       |778            |13033.94    |-4886263.14|-48.19        |
|Pioneer     |Austin      |TX  |10128958.13     |8.75       |739            |13706.30    |-4697526.81|-46.38        |
|Elmwood     |Lakeside    |KY  |10009610.28     |8.65       |771            |12982.63    |-4797793.02|-47.93        |
|Williamstown|State Line  |TX  |9995539.71      |8.64       |755            |13239.13    |-4581982.78|-45.84        |
|Buffalo     |Lewisburg   |KS  |9934051.54      |8.58       |770            |12901.37    |-4745525.30|-47.77        |
|Forest      |Post Oak    |CO  |9905863.64      |8.56   

Gold ca_marge_par_magasin : 12 ligne(s) - granularite : un magasin


## 4. Taux de remise par mois

- Question P2 : « Quel est le niveau moyen des remises, et augmente-t-il avec le temps ? »
- **Granularité** : une ligne = un mois.
- **Règle** : KPI-05, somme des remises ÷ somme des prix catalogue. On calcule le taux sur les sommes, pour ne pas donner le même poids à une petite et à une grosse ligne.

In [7]:
remises = (ratios(ventes.groupBy("annee", "mois").agg(*indicateurs()))
    .select("annee", "mois", "montant_catalogue", "remises", "taux_remise_pct", "nb_lignes")
    .orderBy("annee", "mois"))
remises.show(60)
ventes.groupBy("annee").agg(
    F.round(F.sum("montant_remise") / F.sum("montant_catalogue") * 100, 2).alias("taux_remise_pct")
).orderBy("annee").show()
ecrire_gold(remises, "taux_remise_mensuel", "un mois (annee, mois)")

+-----+----+-----------------+----------+---------------+---------+
|annee|mois|montant_catalogue|   remises|taux_remise_pct|nb_lignes|
+-----+----+-----------------+----------+---------------+---------+
| 2001|   1|       4227304.73|2084244.18|          49.30|     1085|
| 2001|   2|       4014798.62|2038216.61|          50.77|     1053|
| 2001|   3|       4741522.77|2343383.88|          49.42|     1208|
| 2001|   4|       4088287.92|2030857.76|          49.68|     1101|
| 2001|   5|       4349903.40|2284510.00|          52.52|     1098|
| 2001|   6|       4177836.63|2086845.19|          49.95|     1065|
| 2001|   7|       4310533.77|2176004.46|          50.48|     1153|
| 2001|   8|       4269219.38|2107343.49|          49.36|     1072|
| 2001|   9|       4446091.91|2186834.49|          49.19|     1100|
| 2001|  10|       4453531.81|2163539.29|          48.58|     1117|
| 2001|  11|       4458110.40|2261830.29|          50.74|     1184|
| 2001|  12|       4355003.61|2208298.02|       

+-----+---------------+
|annee|taux_remise_pct|
+-----+---------------+
| 2001|          50.05|
| 2002|          50.26|
| 2003|          49.67|
| 2004|          50.05|
| 2005|          49.85|
+-----+---------------+



Gold taux_remise_mensuel : 60 ligne(s) - granularite : un mois (annee, mois)


## 5. Contrôles qualité des résultats Gold

- La somme des CA par catégorie, par magasin et par mois doit être égale au CA total des ventes Silver.
- Il ne doit y avoir aucun indicateur vide.

In [8]:
ca_total = ventes.agg(F.round(F.sum("montant_net_paye"), 2)).first()[0]
print("CA total Silver :", ca_total)
for nom in ["ca_mensuel", "ca_annuel", "ca_marge_par_categorie", "ca_marge_par_magasin"]:
    df = spark.read.parquet(GOLD + "/" + nom)
    somme = df.agg(F.sum("chiffre_affaires")).first()[0]
    vides = df.filter(F.col("chiffre_affaires").isNull() | F.col("marge").isNull()).count()
    print(f"{nom:<24} somme CA = {somme} | ecart = {somme - ca_total} | indicateurs vides = {vides}")
    resultats[nom]["controle_somme_ca"] = str(somme)
    assert abs(somme - ca_total) < 1 and vides == 0

with open(os.path.join(OUTPUTS, "bilan_gold.json"), "w") as f:
    json.dump(resultats, f, indent=1, ensure_ascii=False)

CA total Silver : 115752176.68


ca_mensuel               somme CA = 115752176.68 | ecart = 0.00 | indicateurs vides = 0


ca_annuel                somme CA = 115752176.68 | ecart = 0.00 | indicateurs vides = 0


ca_marge_par_categorie   somme CA = 115752176.68 | ecart = 0.00 | indicateurs vides = 0


ca_marge_par_magasin     somme CA = 115752176.68 | ecart = 0.00 | indicateurs vides = 0


In [9]:
journal("fin | OK")
spark.stop()

2026-11-18 17:31:02 | agregations_gold | fin | OK
